This notebook starts from scratch, loads base, adds rail links, applies base corrections, save out scenario as “corrected base” standard networks





In [1]:
import os
import sys
import numpy as np

from pyproj import CRS
from pathlib import Path

from projectcard import read_cards

import network_wrangler
from network_wrangler import load_roadway
from network_wrangler import load_transit
from network_wrangler import create_scenario
from network_wrangler import Scenario
from network_wrangler.roadway import write_roadway
from network_wrangler.transit import write_transit

from met_council_wrangler import MetCouncil_Parameters
from met_council_wrangler import metcouncil_roadway
from met_council_wrangler import metcouncil_transit

from cube_wrangler import Parameters
from cube_wrangler import util
from cube_wrangler import roadway
from cube_wrangler import StandardTransit

In [2]:
network_wrangler.setup_logging()

In [3]:
%reload_ext autoreload
%autoreload 2

# remote i/o

In [4]:
input_dir = os.path.join(r"Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\standard_networks\v00")
cc_dir = os.path.join(r"Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\standard_networks\v00")
rail_dir = os.path.join(r"Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\standard_networks\v00\v0.5.2")
tran_dir = os.path.join(r"Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\standard_networks\v00\v0.5.2")

net_dir = os.path.join(r"Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\Networks\standard_networks")
output_network = os.path.join(r"Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\output")

metcouncil_wrangler_dir = os.path.join(r"Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\software\met_council_wrangler")
cube_wrangler_dir = os.path.join(r"Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\software\cube_wrangler")

In [5]:
project_card_dir = os.path.join(r"Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\project_card\BaseCorrections_v1")
project_card_dir2 = os.path.join(r"Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\project_card\BaseCorrections2_v1")
project_card_dir3 = os.path.join(r"Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\project_card\BaseMissingRoads_v1")
project_card_dir4 = os.path.join(r"Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\project_card\BaseAttribute_v1")
project_card_dir5 = os.path.join(r"Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\project_card\BaseCorrection_transit_v1")

In [ ]:
metcouncil_parameters = MetCouncil_Parameters(
    metcouncil_wrangler_base_dir=metcouncil_wrangler_dir,
    cube_wrangler_base_dir=cube_wrangler_dir
)

# Load Version00 standard networks

In [ ]:
link_file = os.path.join(input_dir, 'standard_networks', 'links.json')
node_file = os.path.join(input_dir, 'standard_networks', 'nodes.geojson')
shape_file = os.path.join(input_dir, 'standard_networks', 'shapes.geojson')

roadway_net = load_roadway(
    links_file=link_file,
    nodes_file=node_file,
    shapes_file=shape_file,
)

transit_net = load_transit(os.path.join(tran_dir,"standard_transit_network"))

In [ ]:
roadway_net.links_df.shape

In [ ]:
roadway_net.nodes_df.shape

In [10]:
roadway_net.links_df = roadway_net.links_df[roadway_net.links_df.A != roadway_net.links_df.B]

### Attribute the Network

In [11]:
roadway_net.links_df = roadway_net.links_df.drop('lanes', axis = 1)

In [12]:
# make sure the data types of the boolean columns are correct
roadway_net.links_df['bus_only'] = False

for c in list(set(roadway_net.links_df.columns) & set(metcouncil_parameters.bool_col)):
    roadway_net.links_df[c] = roadway_net.links_df[c].replace(
        {
            np.nan: False,
            "": False,
            "0": False,
            "1": True
        }
    )

In [ ]:
r_net = metcouncil_roadway.calculate_number_of_lanes_from_reviewed_network(
    roadway_net=roadway_net,
    parameters=metcouncil_parameters,
)
r_net.links_df.lanes.value_counts()

In [ ]:
r_net.links_df.roadway.value_counts()

In [ ]:
r_net = metcouncil_roadway.calculate_assign_group_and_roadway_class_from_reviewed_network(
        roadway_net=r_net,
        parameters=metcouncil_parameters,
)
r_net.links_df.assign_group.value_counts(dropna=False)

In [ ]:
r_net.links_df.roadway_class.value_counts()

### Add Rail links and nodes

In [ ]:
# rail links and nodes need to be added right after attributing the network and before applying any project cards
# this step can't be moved
r_net = metcouncil_roadway.add_rail_links_and_nodes(
    roadway_network = r_net,
    parameters = metcouncil_parameters,
    rail_links_file = os.path.join(rail_dir, 'rail_links.geojson'),
    rail_nodes_file = os.path.join(rail_dir, 'rail_nodes.geojson'),
)

In [ ]:
# check if missing IDs
# rail nodes does not have osm IDs, they have shst IDs
# rail links does not have osm IDs, they have shst IDs

# if node missing shst id
print(r_net.nodes_df.shst_node_id.isnull().sum())
print(r_net.nodes_df.shst_node_id.nunique())
# if node missing osm id
print(r_net.nodes_df.osm_node_id.isnull().sum() + len(r_net.nodes_df[r_net.nodes_df.osm_node_id==""]))
# if node missing osm id
print(r_net.nodes_df.osm_node_id.dtype)
print(r_net.nodes_df[r_net.nodes_df.osm_node_id == 0])
print(r_net.nodes_df.osm_node_id.nunique())
# if node missing model node id
print(r_net.nodes_df.model_node_id.nunique())

# if link missing 
print(r_net.links_df.shstReferenceId.isnull().sum())
print(r_net.links_df.shstReferenceId.nunique())
print(r_net.links_df.model_link_id.nunique())

# if link missing node id
print(r_net.links_df.fromIntersectionId.isnull().sum())
print(r_net.links_df.toIntersectionId.isnull().sum())
print(r_net.links_df.u.isnull().sum())
print(r_net.links_df[r_net.links_df.u == 0])
print(r_net.links_df.v.isnull().sum())
print(r_net.links_df[r_net.links_df.v == 0])

# Create Scenario 00
A starting point for applying base correction project cards 

In [ ]:
# the new version of transit network (v5) is built on top of v01 roadway network (with all base project cards applied already)
# so exclude transit network in the base_scenario here (otherwise it will fail because the roadway and transit are inconsistent)
# will add the transit network back in v01 scenario later
base_scenario = {"road_net": r_net}
version_00_scenario = create_scenario(base_scenario = base_scenario)

# Create Scenario 00B
This adds BaseAttribute, walk and bike variables to network, manually calculates external connectors, transit priority, and some roadclass values

In [ ]:
version_00b_scenario = create_scenario(
    base_scenario = version_00_scenario,
    project_card_filepath = project_card_dir4
)

In [ ]:
version_00b_scenario.apply_all_projects()

# Create Scenario 00C
This adds BaseCorrections, which correct attributes 

In [ ]:
version_00c_scenario = create_scenario(
    base_scenario=version_00b_scenario,
    project_card_filepath = project_card_dir
)

In [ ]:
version_00c_scenario.add_project_cards(
     list(
            read_cards(project_card_dir3).values()
        )
)

In [ ]:
version_00c_scenario.apply_all_projects()

# Create Version 01
This adds BaseCorrections2 , which are newer attribute clean up cards 

In [ ]:
version_01_scenario = create_scenario(
    base_scenario=version_00c_scenario,
    project_card_filepath = project_card_dir2
)

In [ ]:
version_01_scenario.apply_all_projects()

In [ ]:
version_01_scenario.applied_projects

In [28]:
version_01_scenario.road_net.links_df['access'] = version_01_scenario.road_net.links_df['access'].fillna('')
version_01_scenario.road_net.links_df['access'] = version_01_scenario.road_net.links_df['access'].apply(
    lambda x: util.shorten_name(x)
)

In [29]:
version_01_scenario.road_net.links_df['name'] = version_01_scenario.road_net.links_df['name'].fillna('')
version_01_scenario.road_net.links_df['name'] = version_01_scenario.road_net.links_df['name'].apply(
    lambda x: util.shorten_name(x)
)

# Add Transit to Version 01
This adds transit network to version 01 scenario, and applies transit corrections 

In [30]:
# add transit network to version 01 scenario
version_01_scenario.transit_net = transit_net

In [ ]:
version_01_scenario = create_scenario(
    base_scenario = version_01_scenario,
    project_card_filepath = project_card_dir5, # transit correction
)
version_01_scenario.transit_net.road_net = version_01_scenario.road_net

In [ ]:
version_01_scenario.apply_all_projects()

In [ ]:
version_01_scenario.applied_projects

# Save version 01 standard networks

In [ ]:
version_01_scenario.write(
    os.path.join(net_dir, 'v01', 'standard_networks'),
    name = 'v01',
    roadway_file_format = "geojson",
    transit_file_format = "txt",
    roadway_write = True,
    transit_write = True,
    projects_write = True,
    overwrite = True,
    roadway_convert_complex_link_properties_to_single_field=True
)

# Make Travel Model Network

### Add centroid and centroid connectors

In [ ]:
r_net = metcouncil_roadway.add_centroid_and_centroid_connector(
    roadway_network = version_01_scenario.road_net,
    parameters = metcouncil_parameters,
    centroid_file = os.path.join(input_dir, 'standard_networks', 'centroid_node.pickle'),
    centroid_connector_link_file = os.path.join(input_dir, 'standard_networks', 'cc_link.pickle'),
    centroid_connector_shape_file = os.path.join(input_dir, 'standard_networks', 'cc_shape.pickle'),
)

### Add Rail access and egress links

In [ ]:
r_net = metcouncil_roadway.add_rail_ae_connections(
    r_net,
    metcouncil_parameters,
    exclude_rail_node_id = [417275, 417255]
)

In [ ]:
m_net = metcouncil_roadway.roadway_standard_to_met_council_network(
    r_net,
    metcouncil_parameters    
)

In [ ]:
# check if missing IDs
# centroids does not have osm and shst IDs
# centroid connectors does not have osm and shst IDs

# if node missing shst id
print(m_net.nodes_df.shst_node_id.isnull().sum())
print(m_net.nodes_df.shst_node_id.nunique())

# if node missing model node id
print(m_net.nodes_df.model_node_id.nunique())

# if link missing 
print(m_net.links_df.shstReferenceId.isnull().sum() + len(m_net.links_df[m_net.links_df.shstReferenceId==""]))
print(m_net.links_df.shstReferenceId.nunique())
print(m_net.links_df.model_link_id.nunique())

# if link missing node id
print(m_net.links_df.fromIntersectionId.isnull().sum())
print(m_net.links_df.toIntersectionId.isnull().sum())

In [ ]:
m_net.nodes_df.shape

In [ ]:
m_net.nodes_df.columns

In [ ]:
m_net.links_df.columns

# Write model network as shapefile

In [ ]:
#out_cols = ['model_link_id', 'id', 'assign_group', 'drive_access', 'roadway_class',
#            'lanes_AM', 'lanes_MD', 'lanes_PM', 'lanes_NT', 'segment_id', 'HOV', 'bike', 'walk','roadway',
#            'price_sov_AM', 'geometry', 'managed']

roadway.write_roadway_as_shp(
    roadway_net = m_net,
    parameters = metcouncil_parameters,
    output_link_shp = os.path.join(output_network, 'fullnet_v01', 'shapefile', 'links_v01.shp'),
    output_node_shp = os.path.join(output_network, 'fullnet_v01', 'shapefile', 'nodes_v01.shp'),
    #link_output_variables = out_cols,
    data_to_csv = False,
    data_to_dbf = True,
    export_drive_only = False, # if user only wants drive links/nodes in the shapefile
)

# Write model network for Cube

In [ ]:
roadway.write_roadway_as_fixedwidth(
    roadway_net = m_net,
    parameters = metcouncil_parameters,
    zones = metcouncil_parameters.zones,
    output_link_txt = os.path.join(output_network, 'fullnet_v01', 'links.txt'),
    output_node_txt = os.path.join(output_network,  'fullnet_v01','nodes.txt'),
    output_link_header_width_txt = os.path.join(output_network,  'fullnet_v01', 'links_header_width.txt'),
    output_node_header_width_txt = os.path.join(output_network,  'fullnet_v01','nodes_header_width.txt'),
    output_cube_network_script = os.path.join(output_network,  'fullnet_v01',  'make_complete_network_from_fixed_width_file.s'),
)

In [44]:
version_01_scenario.transit_net.road_net = version_01_scenario.road_net
standard_transit_net = StandardTransit.fromTransitNetwork(version_01_scenario.transit_net, parameters=metcouncil_parameters)

In [ ]:
standard_transit_net = metcouncil_transit.transit_standard_to_met_council_transit_network(
    transit_net = standard_transit_net,
    parameters = metcouncil_parameters,
    line_name_xwalk = os.path.join(output_network, 'fullnet_v01', "line_name_xwalk.csv")
)    

In [46]:
standard_transit_net.write_as_cube_lin(outpath = os.path.join(output_network, 'fullnet_v01', "transit.lin"))

In [ ]:
version_01_scenario.road_net.nodes_df.model_node_id.max()

In [ ]:
version_01_scenario.road_net.nodes_df.model_node_id.shape

In [ ]:
m_net.nodes_df.model_node_id.max()

In [ ]:
m_net.nodes_df.model_node_id.shape